## 检测 length 列异常值

扫描 parquet 文件中 `length` 列，找出与正常值不同的行，记录异常 episode_index。用于数据质量检查，发现帧数不完整的 episode。

**使用示例：**
- 设置 `DATA_DIR` 为 episodes 目录路径（如 `meta/episodes/chunk-000`）
- 设置 `NORMAL_LENGTH` 为预期的正常帧数（如 768）
- 运行后获得异常 episode_index 列表，供后续删除步骤使用

In [3]:
# Jupyter Notebook 单元格代码
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

PROJECT_ROOT = Path.cwd().parent.resolve()
warnings.filterwarnings('ignore')

# ========== 全局变量定义（请根据需求修改）==========
# 数据目录路径（指向包含 parquet 文件的目录）
DATA_DIR = str(PROJECT_ROOT / "datasets/Part_Sorting/auto/episode_500_v2_20260525_164642/meta/episodes/chunk-000")

# 是否保存为txt日志文件（True: 保存, False: 不保存）
SAVE_TO_TXT = True

# txt文件输出目录（设置为当前工作目录）
OUTPUT_DIR = os.getcwd()

# txt文件名前缀（会自动添加时间戳）
TXT_PREFIX = "length_anomaly_log"

# length列的正常值（不等于此值的行将被记录为异常）
NORMAL_LENGTH = 129
# =================================================

# 执行分析
target_dir = Path(DATA_DIR)

# 用于存储输出内容
output_lines = []


def print_and_log(text):
    """同时打印到控制台和日志"""
    print(text)
    if SAVE_TO_TXT:
        output_lines.append(text)


# 开始分析
print_and_log("=" * 100)
print_and_log(f"Length列异常检测报告")
print_and_log(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print_and_log("=" * 100)
print_and_log(f"目标目录: {target_dir}")
print_and_log(f"目录是否存在: {target_dir.exists()}")
print_and_log(f"当前工作目录: {os.getcwd()}")
print_and_log(f"正常length值: {NORMAL_LENGTH}")
print_and_log("=" * 100)

if not target_dir.exists():
    raise FileNotFoundError(f"目录不存在: {target_dir}")

# 递归查找所有 parquet 文件
parquet_files = sorted(target_dir.rglob("*.parquet"))

print_and_log(f"\n共找到 {len(parquet_files)} 个 parquet 文件")

if len(parquet_files) == 0:
    raise FileNotFoundError(f"目录中没有找到任何 parquet 文件: {target_dir}")

# 汇总统计
total_rows = 0
total_length_value_counts = {}  # {length_value: count}
all_anomalies = []  # [(file_path, episode_index, length)]

# 遍历所有parquet文件
print_and_log("\n" + "=" * 100)
print_and_log("逐文件分析")
print_and_log("=" * 100)

for i, file_path in enumerate(parquet_files, 1):
    print_and_log(f"\n[{i}/{len(parquet_files)}] {file_path.relative_to(target_dir)}")
    
    df = pd.read_parquet(file_path)
    total_rows += len(df)
    
    # 检查是否包含 length 和 episode_index 列
    if 'length' not in df.columns:
        print_and_log(f"  ⚠ 跳过: 文件中没有 'length' 列")
        continue
    if 'episode_index' not in df.columns:
        print_and_log(f"  ⚠ 跳过: 文件中没有 'episode_index' 列")
        continue
    
    # 统计 length 列的各值出现次数
    length_counts = df['length'].value_counts().sort_index()
    for val, cnt in length_counts.items():
        total_length_value_counts[int(val)] = total_length_value_counts.get(int(val), 0) + cnt
    
    print_and_log(f"  行数: {len(df)}")
    print_and_log(f"  length 唯一值数量: {df['length'].nunique()}")
    print_and_log(f"  length 值分布: {dict(length_counts)}")
    
    # 找出 length != NORMAL_LENGTH 的行
    anomaly_mask = df['length'] != NORMAL_LENGTH
    anomaly_df = df[anomaly_mask][['episode_index', 'length']]
    
    if len(anomaly_df) > 0:
        print_and_log(f"  🔴 异常行数 (length != {NORMAL_LENGTH}): {len(anomaly_df)}")
        for _, row in anomaly_df.iterrows():
            all_anomalies.append((
                str(file_path.relative_to(target_dir)),
                int(row['episode_index']),
                int(row['length'])
            ))
            print_and_log(f"    episode_index={int(row['episode_index'])}, length={int(row['length'])}")
    else:
        print_and_log(f"  ✅ 所有行的 length 均为 {NORMAL_LENGTH}")

# 汇总报告
print_and_log("\n" + "=" * 100)
print_and_log("汇总报告")
print_and_log("=" * 100)
print_and_log(f"扫描文件数: {len(parquet_files)}")
print_and_log(f"总行数: {total_rows}")
print_and_log(f"")
print_and_log(f"length 列统计:")
print_and_log(f"  唯一值类型数: {len(total_length_value_counts)}")
print_and_log(f"  所有length值及出现次数:")
for val in sorted(total_length_value_counts.keys()):
    cnt = total_length_value_counts[val]
    marker = " ✅ 正常" if val == NORMAL_LENGTH else " 🔴 异常"
    print_and_log(f"    length={val}: {cnt} 行{marker}")

normal_count = total_length_value_counts.get(NORMAL_LENGTH, 0)
anomaly_count = sum(cnt for val, cnt in total_length_value_counts.items() if val != NORMAL_LENGTH)
print_and_log(f"")
print_and_log(f"正常行数 (length={NORMAL_LENGTH}): {normal_count}")
print_and_log(f"异常行数 (length!={NORMAL_LENGTH}): {anomaly_count}")
print_and_log(f"异常 episode_index 总数: {len(all_anomalies)}")

if all_anomalies:
    print_and_log(f"\n异常 episode_index 列表:")
    for file_name, ep_idx, length_val in all_anomalies:
        print_and_log(f"  episode_index={ep_idx}, length={length_val}, 文件={file_name}")

# 保存为txt文件
if SAVE_TO_TXT:
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    txt_filename = f"{TXT_PREFIX}_{timestamp}.txt"
    txt_filepath = output_dir / txt_filename
    
    with open(txt_filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))
    
    print_and_log("\n" + "=" * 100)
    print_and_log(f"日志文件已保存至: {txt_filepath}")
    print_and_log(f"保存目录: {output_dir.absolute()}")
    print_and_log("=" * 100)
else:
    print_and_log("\n" + "=" * 100)
    print_and_log("未保存txt文件（SAVE_TO_TXT = False）")
    print_and_log("=" * 100)

# 单独输出异常 episode_index 为纯列表（方便后续使用）
if all_anomalies:
    anomaly_ep_ids = sorted(set(ep_idx for _, ep_idx, _ in all_anomalies))
    print_and_log(f"\n异常 episode_index 去重列表 (共 {len(anomaly_ep_ids)} 个):")
    print_and_log(f"{anomaly_ep_ids}")

print_and_log("\n分析完成！")

Length列异常检测报告
生成时间: 2026-05-26 10:33:35
目标目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Part_Sorting/auto/episode_500_v2_20260525_164642/meta/episodes/chunk-000
目录是否存在: True
当前工作目录: /data/SJJ/UBT/lerobot_0.5.1/dataset_post_process
正常length值: 129

共找到 1 个 parquet 文件

逐文件分析

[1/1] file-000.parquet
  行数: 500
  length 唯一值数量: 1
  length 值分布: {129: np.int64(500)}
  ✅ 所有行的 length 均为 129

汇总报告
扫描文件数: 1
总行数: 500

length 列统计:
  唯一值类型数: 1
  所有length值及出现次数:
    length=129: 500 行 ✅ 正常

正常行数 (length=129): 500
异常行数 (length!=129): 0
异常 episode_index 总数: 0

日志文件已保存至: /data/SJJ/UBT/lerobot_0.5.1/dataset_post_process/length_anomaly_log_20260526_103335.txt
保存目录: /data/SJJ/UBT/lerobot_0.5.1/dataset_post_process

分析完成！


## 调用 lerobot_edit_dataset 删除异常 episode

使用 LeRobot 官方的 `lerobot_edit_dataset.py` 脚本，根据上一步检测出的异常 episode_index 列表，从数据集中删除对应的 episode。支持原地修改（备份原数据）或输出到新目录。

**使用示例：**
- 将上一步输出的异常 episode_index 列表填入 `ANOMALY_EPISODE_INDICES`
- 设置 `DATASET_ROOT` 为数据集根目录（包含 `meta/` 和 `data/` 的目录）
- 首次运行建议设置 `IN_PLACE = False`，先输出到新目录验证结果
- 确认无误后再设置 `IN_PLACE = True` 原地修改

In [7]:
# 调用 lerobot_edit_dataset 删除异常 episode

import subprocess
import sys
import os
from pathlib import Path

# ========== 配置参数（请根据需求修改）==========
# 仓库根目录
REPO_ROOT = Path.cwd().parent.resolve()

# 数据集根目录（包含 meta/ 和 data/ 的目录）
DATASET_ROOT = REPO_ROOT / "datasets/Packing_Box/auto/Packing_Box_episode_1397"


NEW_DATASET_ROOT = REPO_ROOT / "datasets/Packing_Box/auto/Packing_Box_episode_1397_filtered"
# 数据集的 repo_id（任意标识名）
REPO_ID = "Packing_Box_episode_1397"

# 异常 episode_index 列表（从上一步分析日志获取）
ANOMALY_EPISODE_INDICES = [
    89, 121, 129, 137, 138, 167, 179, 184, 187, 201,
    206, 208, 310, 318, 332, 336, 342, 356, 357, 382,
    414, 878, 879, 884, 894, 914, 926, 1390, 1391, 1396
]

# 是否原地修改（True: 原地删除，原数据备份到 xxx_old; False: 输出到新目录）
IN_PLACE = True

# 新数据集输出路径（仅当 IN_PLACE=False 时使用）
NEW_ROOT = REPO_ROOT / "datasets/Packing_Box/auto/Packing_Box_episode_1397_filtered"
# =================================================

episode_indices_str = str(ANOMALY_EPISODE_INDICES)

cmd = [
    sys.executable, os.path.join(str(REPO_ROOT), "src/lerobot/scripts/lerobot_edit_dataset.py"),
    "--repo_id", REPO_ID,
    "--root", str(DATASET_ROOT),
    "--new_root", str(NEW_DATASET_ROOT),  # 必须传入，否则默认落到 ~/.cache/... 下
    "--operation.type", "delete_episodes",
    "--operation.episode_indices", episode_indices_str,
]

if not IN_PLACE:
    cmd[cmd.index("--new_root") + 1] = str(NEW_ROOT)
    cmd.insert(cmd.index("--operation.type"), "--new_repo_id")
    cmd.insert(cmd.index("--operation.type"), f"{REPO_ID}_filtered")

# 将仓库根目录加入 PYTHONPATH，使脚本中的 from src.lerobot... 能正常导入
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT) + (
    ":" + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)

print(f"数据集根目录: {DATASET_ROOT}")
print(f"将删除 {len(ANOMALY_EPISODE_INDICES)} 个异常 episode:")
print(f"  {ANOMALY_EPISODE_INDICES}")
if not IN_PLACE:
    print(f"输出到新目录: {NEW_ROOT}")
else:
    print(f"原地修改，原数据将备份到: {DATASET_ROOT}_old")
print(f"\n执行命令:")
print(f"  {' '.join(cmd)}\n")

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_ROOT), env=env)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print(f"\n✅ 成功删除 {len(ANOMALY_EPISODE_INDICES)} 个异常 episode!")
else:
    print(f"\n❌ 命令执行失败，returncode={result.returncode}")


数据集根目录: /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397
将删除 30 个异常 episode:
  [89, 121, 129, 137, 138, 167, 179, 184, 187, 201, 206, 208, 310, 318, 332, 336, 342, 356, 357, 382, 414, 878, 879, 884, 894, 914, 926, 1390, 1391, 1396]
原地修改，原数据将备份到: /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397_old

执行命令:
  /data/anaconda3/envs/lerobot/bin/python /data/SJJ/UBT/lerobot_0.5.1/src/lerobot/scripts/lerobot_edit_dataset.py --repo_id Packing_Box_episode_1397 --root /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397 --new_root /data/SJJ/UBT/lerobot_0.5.1/datasets/Packing_Box/auto/Packing_Box_episode_1397_filtered --operation.type delete_episodes --operation.episode_indices [89, 121, 129, 137, 138, 167, 179, 184, 187, 201, 206, 208, 310, 318, 332, 336, 342, 356, 357, 382, 414, 878, 879, 884, 894, 914, 926, 1390, 1391, 1396]


INFO 2026-05-23 22:37:03 eo_utils.py:106 Using video codec: libsvtav1
INFO 2026-05-2

## 将长 episode 按 part_info 拆分为短 episode

根据 `meta/part_info.json` 中记录的零件帧范围，将一个包含多个零件操作的长 episode 拆分为多个独立的短 episode。支持保留或移除视频观测数据，支持单数据集和批量目录两种处理模式。

> **权限问题说明**：`LeRobotDataset.create()` 会向上递归 `chmod 777` 父目录。当输出目录的父目录属于 `root` 时，会抛出 `PermissionError`。本单元格已内置 monkey-patch 绕过该问题。

**使用示例：**
- 单数据集模式：设置 `PROCESS_MODE = "single_dataset"`，指定 `SOURCE_DATASET_ROOT`
- 目录模式：设置 `PROCESS_MODE = "directory"`，批量处理目录下所有数据集
- 设置 `PRESERVE_VISUAL_OBS = True` 保留视频帧，`False` 仅保留数值数据（处理更快）
- 设置 `OVERWRITE_OUTPUT = False` 防止意外覆盖已有输出目录

> 也可以用独立脚本运行：`python datasets/split_conveyor_episodes.py`

In [ ]:
# 将 Conveyor_Sorting 长 episode 按 meta/part_info.json 拆分为单零件短 episode
# 纯离线处理：不启动 Isaac Sim，只读取源 LeRobotDataset 的 observation/action，再写出新 LeRobotDataset。

from __future__ import annotations

import copy
import json
import shutil
from pathlib import Path

import numpy as np

import os
import sys

REPO_ROOT = Path.cwd().parent.resolve()

# Jupyter 当前 kernel 的 import 搜索路径需要直接改 sys.path；
# 只修改 env 字典只会影响 subprocess，不会影响本单元的 from src... import。
repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)
os.environ["PYTHONPATH"] = repo_root_str + (
    ":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

# Hugging Face datasets 会创建 cache lock 文件。默认 ~/.cache 在部分环境中可能是只读，
# 因此在导入 LeRobotDataset 前把 cache 放到仓库内可写目录。
HF_CACHE_ROOT = REPO_ROOT / ".cache" / "huggingface"
os.environ.setdefault("HF_HOME", str(HF_CACHE_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE_ROOT / "datasets"))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE_ROOT / "hub"))
(HF_CACHE_ROOT / "datasets").mkdir(parents=True, exist_ok=True)
(HF_CACHE_ROOT / "hub").mkdir(parents=True, exist_ok=True)

from src.lerobot.datasets.lerobot_dataset import LeRobotDataset

# ── Monkey-patch: 绕过 os.chmod 权限问题 ──────────────────────
# LeRobotDataset.create() 向上递归 chmod 777 父目录，
# 当父目录属于 root 时会抛出 PermissionError。
_original_chmod = os.chmod


def _noop_chmod(path, mode, *args, **kwargs):
    try:
        _original_chmod(path, mode, *args, **kwargs)
    except PermissionError:
        pass  # 无权限修改时静默跳过


os.chmod = _noop_chmod

# ========== 全局变量定义（请根据需求修改）==========

# 处理模式：
# - "single_dataset": 只处理 SOURCE_DATASET_ROOT 指向的一个 LeRobot 数据集
# - "directory": 扫描 SOURCE_DATASET_DIR 下所有包含 meta/info.json 和 meta/part_info.json 的 LeRobot 数据集
PROCESS_MODE = "single_dataset"

# 单数据集模式：源数据集根目录，必须包含 meta/、data/，以及 meta/part_info.json
SOURCE_DATASET_ROOT = REPO_ROOT / "datasets/Conveyor_Sorting/auto/test"

# 目录模式：扫描该目录下所有 LeRobot 数据集
SOURCE_DATASET_DIR = REPO_ROOT / "datasets/Conveyor_Sorting/auto"
DISCOVER_RECURSIVE = True

# 输出位置：
# - 单数据集模式下，OUTPUT_DATASET_ROOT 是新数据集根目录
# - 目录模式下，每个源数据集会输出到 OUTPUT_PARENT_DIR / f"{源数据集目录名}{OUTPUT_NAME_SUFFIX}"
OUTPUT_DATASET_ROOT = REPO_ROOT / "datasets/Conveyor_Sorting/auto/test_short_parts"
OUTPUT_PARENT_DIR = REPO_ROOT / "datasets/Conveyor_Sorting/auto_short_parts"
OUTPUT_NAME_SUFFIX = "_short_parts"

# repo_id 只写入本地 meta，可自定义；为空时自动用源目录名 + OUTPUT_NAME_SUFFIX
OUTPUT_REPO_ID = "Conveyor_Sorting_test_short_parts"
OUTPUT_REPO_ID_SUFFIX = "_short_parts"

# 是否保留视频/图像观测。
# True: 通过 source_dataset[abs_idx] 解码视频帧并重新编码到新数据集，最完整但较慢。
# False: 移除 video/image 特征，仅保留 action、observation.state 等非图像数据，速度更快。
PRESERVE_VISUAL_OBS = True

# 视频解码后端。
# 仍然使用 LeRobotDataset.__getitem__() 官方视频读取路径；这里只指定官方支持的 pyav backend。
# 不指定时会默认优先 torchcodec，但当前 torchcodec 版本不接受 fsspec LocalFileOpener。
SOURCE_VIDEO_BACKEND = "pyav"

# 输出视频编码器。LeRobotDataset.save_episode() 会用该 codec 将 add_frame 写出的临时 PNG 编码为 mp4。
OUTPUT_VIDEO_CODEC = "libsvtav1"

# 是否允许删除已有输出目录
OVERWRITE_OUTPUT = False

# 新短 episode 的 task 文本。None 表示沿用源数据集 task。
SINGLE_TASK = None

# =================================================


DEFAULT_FRAME_KEYS = {"index", "episode_index", "frame_index", "timestamp", "task_index"}


def _dataset_has_part_info(dataset_root: Path) -> bool:
    return (dataset_root / "meta" / "info.json").exists() and (dataset_root / "meta" / "part_info.json").exists()


def discover_source_datasets() -> list[Path]:
    if PROCESS_MODE == "single_dataset":
        return [Path(SOURCE_DATASET_ROOT)]
    if PROCESS_MODE != "directory":
        raise ValueError(f"Unsupported PROCESS_MODE: {PROCESS_MODE!r}")

    base = Path(SOURCE_DATASET_DIR)
    info_paths = base.rglob("meta/info.json") if DISCOVER_RECURSIVE else base.glob("*/meta/info.json")
    roots = sorted({path.parent.parent for path in info_paths})
    return [root for root in roots if _dataset_has_part_info(root)]


def make_output_root(source_root: Path) -> Path:
    if PROCESS_MODE == "single_dataset":
        return Path(OUTPUT_DATASET_ROOT)
    return Path(OUTPUT_PARENT_DIR) / f"{source_root.name}{OUTPUT_NAME_SUFFIX}"


def make_output_repo_id(source_root: Path) -> str:
    if PROCESS_MODE == "single_dataset" and OUTPUT_REPO_ID:
        return OUTPUT_REPO_ID
    return f"{source_root.name}{OUTPUT_REPO_ID_SUFFIX}"


def load_part_info(source_root: Path) -> list[dict]:
    path = source_root / "meta" / "part_info.json"
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    episodes = data.get("episodes", data if isinstance(data, list) else None)
    if not isinstance(episodes, list):
        raise ValueError(f"Invalid part_info format: {path}")
    return episodes


def output_features_from_source(source_dataset: LeRobotDataset) -> dict:
    features = copy.deepcopy(source_dataset.meta.features)
    if PRESERVE_VISUAL_OBS:
        return features
    return {
        key: value
        for key, value in features.items()
        if value.get("dtype") not in {"image", "video"}
    }


def as_numpy(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    if hasattr(value, "cpu") and hasattr(value, "numpy"):
        return value.cpu().numpy()
    if isinstance(value, np.ndarray):
        return value
    return np.asarray(value)


def normalize_feature_value(key: str, value, feature: dict):
    dtype = feature.get("dtype")
    if dtype in {"image", "video"}:
        array = as_numpy(value)
        # LeRobot add_frame validates the HWC shape used in this dataset's meta/info.json.
        expected_shape = tuple(feature.get("shape", ()))
        if array.ndim == 3 and expected_shape and tuple(array.shape) != expected_shape:
            if array.shape[0] in (1, 3, 4) and len(expected_shape) == 3 and expected_shape[-1] in (1, 3, 4):
                array = np.moveaxis(array, 0, -1)
        if array.dtype != np.uint8:
            if np.issubdtype(array.dtype, np.floating) and array.max(initial=0) <= 1.0:
                array = np.clip(array * 255.0, 0, 255).astype(np.uint8)
            else:
                array = array.astype(np.uint8)
        return array

    if dtype == "string":
        return str(value)

    array = as_numpy(value)
    if dtype:
        array = array.astype(np.dtype(dtype), copy=False)
    return array


def build_absolute_to_row_index(source_dataset: LeRobotDataset) -> dict[int, int]:
    return {
        int(as_numpy(abs_idx).reshape(-1)[0]): row_idx
        for row_idx, abs_idx in enumerate(source_dataset.hf_dataset["index"])
    }


def get_source_frame(
    source_dataset: LeRobotDataset,
    absolute_frame_index: int,
    include_visual: bool,
    absolute_to_row_index: dict[int, int],
) -> dict:
    row_index = absolute_to_row_index.get(int(absolute_frame_index))
    if row_index is None:
        raise IndexError(f"Global frame index {absolute_frame_index} not found in source dataset")
    if include_visual:
        return source_dataset[row_index]
    return source_dataset.hf_dataset[row_index]


def build_output_frame(source_item: dict, output_features: dict, task: str) -> dict:
    frame = {"task": task}
    for key, feature in output_features.items():
        if key in DEFAULT_FRAME_KEYS:
            continue
        if key not in source_item:
            raise KeyError(f"Source frame missing feature {key!r}")
        frame[key] = normalize_feature_value(key, source_item[key], feature)
    return frame


def episode_task(source_dataset: LeRobotDataset, source_item: dict) -> str:
    if SINGLE_TASK is not None:
        return SINGLE_TASK
    task = source_item.get("task")
    if task is not None:
        return str(task)
    task_index = source_item.get("task_index")
    if task_index is not None:
        task_index = int(as_numpy(task_index).reshape(-1)[0])
        return str(source_dataset.meta.tasks.iloc[task_index].name)
    if len(source_dataset.meta.tasks) > 0:
        return str(source_dataset.meta.tasks.index[0])
    return source_dataset.repo_id


def validate_part_range(source_dataset: LeRobotDataset, source_episode: dict, part: dict) -> tuple[int, int]:
    episode_index = int(source_episode["episode_index"])
    ep_meta = source_dataset.meta.episodes[episode_index]
    episode_length = int(source_episode.get("episode_frame_length", ep_meta["length"]))

    start = int(part["frame_start_index"])
    end = int(part["frame_end_index"])
    if start < 0 or end < start or end >= episode_length:
        raise ValueError(
            f"Invalid frame range for episode {episode_index}, part {part.get('part_name')}: {start}-{end}, "
            f"episode_length={episode_length}"
        )
    return start, end


def split_one_dataset(source_root: Path, output_root: Path, output_repo_id: str) -> dict:
    source_root = Path(source_root)
    output_root = Path(output_root)

    if not _dataset_has_part_info(source_root):
        raise FileNotFoundError(f"Missing meta/info.json or meta/part_info.json under {source_root}")

    if output_root.exists():
        if not OVERWRITE_OUTPUT:
            raise FileExistsError(f"Output already exists: {output_root}. Set OVERWRITE_OUTPUT=True to replace it.")
        shutil.rmtree(output_root)

    print(f"\n=== Split dataset ===")
    print(f"source: {source_root}")
    print(f"output: {output_root}")

    source_dataset = LeRobotDataset(
        repo_id=source_root.name,
        root=source_root,
        video_backend=SOURCE_VIDEO_BACKEND,
    )
    part_info_episodes = load_part_info(source_root)
    absolute_to_row_index = build_absolute_to_row_index(source_dataset)
    output_features = output_features_from_source(source_dataset)
    use_videos = any(feature.get("dtype") == "video" for feature in output_features.values())

    output_dataset = LeRobotDataset.create(
        repo_id=output_repo_id,
        fps=source_dataset.fps,
        features=output_features,
        root=output_root,
        robot_type=source_dataset.meta.robot_type,
        use_videos=use_videos,
        video_backend=source_dataset.video_backend,
        vcodec=OUTPUT_VIDEO_CODEC,
        image_writer_processes=0,
        image_writer_threads=0,
        streaming_encoding=False,
        metadata_buffer_size=1,
    )

    created_episodes = 0

    try:
        for source_episode in part_info_episodes:
            source_episode_index = int(source_episode["episode_index"])
            ep_meta = source_dataset.meta.episodes[source_episode_index]
            episode_from_index = int(ep_meta["dataset_from_index"])

            for part_idx, part in enumerate(source_episode.get("parts", [])):
                start, end = validate_part_range(source_dataset, source_episode, part)
                output_dataset.clear_episode_buffer(delete_images=False)

                first_abs_idx = episode_from_index + start
                first_item = get_source_frame(
                    source_dataset, first_abs_idx, PRESERVE_VISUAL_OBS, absolute_to_row_index
                )
                task = episode_task(source_dataset, first_item)

                for local_frame_idx in range(start, end + 1):
                    abs_idx = episode_from_index + local_frame_idx
                    source_item = first_item if abs_idx == first_abs_idx else get_source_frame(
                        source_dataset, abs_idx, PRESERVE_VISUAL_OBS, absolute_to_row_index
                    )
                    output_dataset.add_frame(build_output_frame(source_item, output_features, task))

                new_episode_index = int(output_dataset.episode_buffer["episode_index"])
                new_episode_length = int(output_dataset.episode_buffer["size"])
                output_dataset.save_episode()

                created_episodes += 1
                print(
                    f"  new_ep={new_episode_index:04d} "
                    f"source_ep={source_episode_index} part={part.get('part_name')} "
                    f"frames={start}-{end} length={new_episode_length}"
                )
    finally:
        output_dataset.finalize()

    print(f"created short episodes: {created_episodes}")
    return {
        "source_root": str(source_root),
        "output_root": str(output_root),
        "created_episodes": created_episodes,
    }


source_roots = discover_source_datasets()
if not source_roots:
    raise RuntimeError("No source LeRobot datasets with meta/part_info.json found.")

results = []
for source_root in source_roots:
    results.append(
        split_one_dataset(
            source_root=source_root,
            output_root=make_output_root(source_root),
            output_repo_id=make_output_repo_id(source_root),
        )
    )

print("\n=== Done ===")
for result in results:
    print(result)


